# CBT Cognitive Distortion Classification with DeBERTa + HQDE

**Optimized for Kaggle: 2x T4 GPUs, 4 vCPUs**

This notebook trains an ensemble of DeBERTa models using the HQDE framework to classify cognitive distortions in therapy conversations.

## Hardware Configuration
- **GPUs**: 2x T4
- **vCPUs**: 4
- **RAM**: ~30GB

## Expected Results
- **Training Time**: ~30-45 minutes
- **Expected Accuracy**: 85-90%

## Cognitive Distortion Classes (10)
0. All-or-Nothing Thinking
1. Overgeneralization
2. Mental Filter
3. Disqualifying the Positive
4. Jumping to Conclusions
5. Magnification/Catastrophizing
6. Emotional Reasoning
7. Should Statements
8. Labeling
9. Personalization

In [ ]:
# Install HQDE package and notebook dependencies
# --no-deps keeps Kaggle's preinstalled CUDA/Torch stack intact.
!pip install -q --no-deps "hqde==0.1.12"
!pip install -q "transformers>=4.30.0" "datasets>=2.14.0" "accelerate>=0.20.0" "sentencepiece>=0.1.99"
!pip install -q "scikit-learn>=1.3.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "seaborn>=0.12.0"

In [ ]:
import os
import json
import random
from contextlib import nullcontext
from importlib.metadata import PackageNotFoundError, version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    from torch.amp import autocast, GradScaler
except ImportError:  # Older PyTorch fallback
    from torch.cuda.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    get_cosine_schedule_with_warmup,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

try:
    print(f"HQDE package version: {version('hqde')}")
except PackageNotFoundError:
    print("HQDE package is not installed. Run the install cell first.")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No CUDA GPU found. The notebook will run in CPU-compatible mode for smoke tests.")

In [ ]:
# Set random seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# Configuration
class Config:
    # Runtime knobs
    quick_test = os.environ.get("HQDE_QUICK_TEST", "0") == "1"
    detected_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    num_gpus = min(detected_gpus, 2)
    device = "cuda" if num_gpus > 0 else "cpu"

    # Model
    model_name = os.environ.get("HQDE_MODEL_NAME", "microsoft/deberta-v3-base")
    num_classes = 10
    max_length = 64 if quick_test else 256

    # Training
    num_workers = 1 if quick_test or num_gpus == 0 else min(4, num_gpus * 2)
    batch_size = 2 if quick_test or num_gpus == 0 else 8
    num_epochs = 1 if quick_test else 15
    learning_rate = 2e-5
    weight_decay = 0.01
    warmup_ratio = 0.1
    max_grad_norm = 1.0

    # Ensemble diversity
    dropout_rates = [0.1, 0.15, 0.2, 0.25]
    learning_rates = [1.5e-5, 2e-5, 2.5e-5, 3e-5]

    # DataLoader/runtime
    dataloader_workers = 0 if quick_test or os.name == "nt" else 2
    pin_memory = num_gpus > 0
    use_amp = num_gpus > 0

    # Data
    test_size = 0.2
    val_size = 0.1

    # Output
    output_dir = "./cbt_output"

    # Class names
    distortion_names = [
        "All-or-Nothing Thinking",
        "Overgeneralization",
        "Mental Filter",
        "Disqualifying the Positive",
        "Jumping to Conclusions",
        "Magnification/Catastrophizing",
        "Emotional Reasoning",
        "Should Statements",
        "Labeling",
        "Personalization",
    ]

config = Config()
os.makedirs(config.output_dir, exist_ok=True)

print("Configuration set")
print(f"  Model: {config.model_name}")
print(f"  Runtime: device={config.device}, detected_gpus={config.detected_gpus}, active_gpus={config.num_gpus}")
print(f"  Ensemble workers: {config.num_workers}")
print(f"  Batch size per worker: {config.batch_size}")
print(f"  Epochs: {config.num_epochs}")
print(f"  AMP enabled: {config.use_amp}")
print(f"  DataLoader workers: {config.dataloader_workers}")
if config.quick_test:
    print("  Quick test mode enabled via HQDE_QUICK_TEST=1")

In [ ]:
# Generate CBT dataset
def generate_cbt_dataset():
    """Generate 100 CBT therapy conversations."""
    conversations = []
    conv_id = 1
    
    templates = {
        0: [  # All-or-Nothing
            ("I got a B+ on my exam. I'm devastated. I studied so hard, but it wasn't enough. I'm either an A student or I'm a complete failure. There's no in-between for me. Because it's not perfect! If I can't get an A, then what's the point? I either do things perfectly or I might as well not do them at all.", "frustrated, anxious"),
            ("My presentation went well, but I stumbled over one word. That one mistake ruined the whole thing. It's either perfect or it's garbage. No, you don't understand. If it's not flawless, then it's worthless.", "disappointed, perfectionistic"),
        ],
        1: [  # Overgeneralization
            ("I asked someone out and they said no. This always happens to me. I'm never going to find a partner. Every time I try, I get rejected. Well, this was only the second time, but both times they said no. So clearly I'm just not dateable. Everyone rejects me.", "hopeless, defeated"),
            ("I made a mistake at work today. I always mess things up. I never do anything right. This is typical of me. It happens all the time. Every project I touch goes wrong.", "defeated, hopeless"),
        ],
        2: [  # Mental Filter
            ("My performance review was mostly positive, but there was one area for improvement mentioned. That's the only part I can think about. The positive feedback doesn't matter. That one criticism is all I hear. I keep replaying it in my mind.", "anxious, dissatisfied"),
            ("I had a great day, but I made one small social mistake. That mistake is all I remember now. Everything else fades away compared to that one error.", "regretful, fixated"),
        ],
        3: [  # Disqualifying Positive
            ("I got promoted, but it doesn't mean anything. They probably just needed someone and I was there. It's not like I earned it. Yes, I exceeded the metrics, but that doesn't really count. The metrics were probably easy.", "doubtful, unworthy"),
            ("People complimented my work, but they were probably just being polite. They don't really mean it. It's not genuine praise.", "dismissive, skeptical"),
        ],
        4: [  # Jumping to Conclusions
            ("I saw my coworkers whispering. They were definitely talking about me. They must think I'm incompetent. No, I didn't ask. It was obvious from their body language. My manager has been quiet - he's probably building a case to fire me.", "paranoid, anxious"),
            ("My friend hasn't texted me back in two hours. She must be mad at me. She's probably ending our friendship. I just know it.", "worried, insecure"),
        ],
        5: [  # Catastrophizing
            ("I have a headache and I'm tired. This could be a brain tumor. I might be dying. My whole life could be over. I'm terrified. This is a disaster. I can't stop thinking about worst-case scenarios.", "panicked, terrified"),
            ("I made a small mistake in my email. This is a complete disaster. Everyone will think I'm unprofessional. My career is ruined.", "overwhelmed, panicked"),
        ],
        6: [  # Emotional Reasoning
            ("I feel stupid, so I must be stupid. I feel like everyone thinks I'm incompetent, so I must be incompetent. I feel it so strongly. If I feel anxious, it means something bad will happen. My emotions are telling me the truth.", "insecure, anxious"),
            ("I feel like a fraud at work, so I must be a fraud. My feelings don't lie to me. I feel overwhelmed, which means I can't handle my life.", "fraudulent, overwhelmed"),
        ],
        7: [  # Should Statements
            ("I should be able to handle everything. I must be strong all the time. I ought to be perfect at my job, as a parent, as a partner. I should never burden others. I must always be productive. If I don't meet these standards, I feel guilty.", "guilty, stressed, exhausted"),
            ("I must never make mistakes. I should be perfect in everything I do. I ought to exceed everyone's expectations always. I should always put others first.", "pressured, exhausted"),
        ],
        8: [  # Labeling
            ("I made a mistake - I sent an email to the wrong person. I'm such an idiot. I'm a complete failure. I'm worthless. That's what I am. I made a mistake, so I'm a screw-up. I'm fundamentally flawed.", "ashamed, self-loathing"),
            ("I forgot someone's name, so I'm a terrible person. One social mistake makes me fundamentally flawed. I'm just a loser.", "embarrassed, self-hating"),
        ],
        9: [  # Personalization
            ("My team didn't meet the deadline. It's all my fault. If only I had worked harder, this wouldn't have happened. I'm responsible for everyone's failure. There were eight of us, but I should have done more. Their failures are my failures.", "guilty, ashamed"),
            ("My friend is upset. I must have done something to cause it. Everything bad that happens around me is because of me. It's my responsibility to fix their mood.", "guilty, responsible"),
        ]
    }
    
    for distortion_type in range(10):
        for i in range(10):
            template_idx = i % len(templates[distortion_type])
            text, emotion = templates[distortion_type][template_idx]
            
            conversations.append({
                "id": conv_id,
                "text": text,
                "label": distortion_type,
                "emotion": emotion,
                "distortion_name": config.distortion_names[distortion_type]
            })
            conv_id += 1
    
    return conversations

conversations = generate_cbt_dataset()
df = pd.DataFrame(conversations)

print(f"✓ Dataset created: {len(df)} conversations")
print(f"  Classes: {df['label'].nunique()}")
print(f"\nClass Distribution:")
for label in sorted(df['label'].unique()):
    count = (df['label'] == label).sum()
    name = config.distortion_names[label]
    print(f"  {label}: {name:35s} - {count} samples")

In [ ]:
# Split data
train_df, temp_df = train_test_split(
    df, 
    test_size=config.test_size + config.val_size,
    random_state=42,
    stratify=df['label']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=config.test_size / (config.test_size + config.val_size),
    random_state=42,
    stratify=temp_df['label']
)

print(f"✓ Data split:")
print(f"  Training: {len(train_df)} samples")
print(f"  Validation: {len(val_df)} samples")
print(f"  Test: {len(test_df)} samples")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
print(f"\n✓ Tokenizer loaded: {config.model_name}")

# Dataset class
class CBTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = CBTDataset(train_df['text'].values, train_df['label'].values, tokenizer, config.max_length)
val_dataset = CBTDataset(val_df['text'].values, val_df['label'].values, tokenizer, config.max_length)
test_dataset = CBTDataset(test_df['text'].values, test_df['label'].values, tokenizer, config.max_length)

# Create data loaders
loader_kwargs = {
    'num_workers': config.dataloader_workers,
    'pin_memory': config.pin_memory,
}

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size * 2, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size * 2, shuffle=False, **loader_kwargs)

print(f"✓ Datasets and loaders created")

In [ ]:
# DeBERTa Classifier
class DeBERTaClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout_rate=0.1):
        super(DeBERTaClassifier, self).__init__()
        
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.hidden_dropout_prob = dropout_rate
        self.config.attention_probs_dropout_prob = dropout_rate
        self.config.torch_dtype = torch.float32
        
        # Keep trainable parameters in FP32. AMP autocast will use FP16 for
        # selected CUDA operations, while GradScaler expects FP32 gradients.
        self.deberta = AutoModel.from_pretrained(
            model_name,
            config=self.config,
            torch_dtype=torch.float32,
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.config.hidden_size, num_classes)
        
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)
        self.float()
    
    def forward(self, input_ids, attention_mask):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

print("✓ Model architecture defined")

In [ ]:
# Ensemble Worker
def make_grad_scaler(enabled: bool):
    try:
        return GradScaler("cuda", enabled=enabled)
    except TypeError:
        return GradScaler(enabled=enabled)


def amp_context(device: torch.device, enabled: bool):
    if not enabled:
        return nullcontext()
    try:
        return autocast(device_type=device.type, dtype=torch.float16, enabled=enabled)
    except TypeError:
        return autocast(enabled=enabled)


class EnsembleWorker:
    def __init__(self, worker_id, config, device):
        self.worker_id = worker_id
        self.config = config
        self.device = torch.device(device)

        self.dropout_rate = config.dropout_rates[worker_id % len(config.dropout_rates)]
        self.learning_rate = config.learning_rates[worker_id % len(config.learning_rates)]
        self.use_amp = bool(config.use_amp and self.device.type == "cuda")

        self.model = DeBERTaClassifier(config.model_name, config.num_classes, self.dropout_rate).to(self.device)
        if self.use_amp:
            self.model.float()
        first_param = next(self.model.parameters())
        print(
            f"Worker {worker_id}: dropout={self.dropout_rate}, lr={self.learning_rate}, "
            f"device={self.device}, amp={self.use_amp}, param_dtype={first_param.dtype}"
        )

        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.learning_rate,
            weight_decay=config.weight_decay,
        )
        self.criterion = nn.CrossEntropyLoss()
        self.scaler = make_grad_scaler(self.use_amp)

        self.train_losses = []
        self.val_losses = []
        self.val_accuracies = []

    def train_epoch(self, train_loader, scheduler):
        self.model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f"Worker {self.worker_id} Training")
        for batch in pbar:
            input_ids = batch['input_ids'].to(self.device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(self.device, non_blocking=True)
            labels = batch['labels'].to(self.device, non_blocking=True)

            self.optimizer.zero_grad(set_to_none=True)

            with amp_context(self.device, self.use_amp):
                logits = self.model(input_ids, attention_mask)
                loss = self.criterion(logits, labels)

            if self.use_amp:
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)
                self.optimizer.step()

            scheduler.step()

            batch_size = labels.size(0)
            total_loss += loss.detach().item() * batch_size
            _, predicted = torch.max(logits.detach(), 1)
            total += batch_size
            correct += (predicted == labels).sum().item()

            pbar.set_postfix({'loss': f'{loss.detach().item():.4f}', 'acc': f'{100 * correct / max(total, 1):.2f}%'})

        return total_loss / max(total, 1), 100 * correct / max(total, 1)

    def evaluate(self, val_loader):
        self.model.eval()
        total_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Worker {self.worker_id} Validation"):
                input_ids = batch['input_ids'].to(self.device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(self.device, non_blocking=True)
                labels = batch['labels'].to(self.device, non_blocking=True)

                logits = self.model(input_ids, attention_mask)
                loss = self.criterion(logits, labels)

                batch_size = labels.size(0)
                total_loss += loss.detach().item() * batch_size
                _, predicted = torch.max(logits, 1)
                total += batch_size
                correct += (predicted == labels).sum().item()

        return total_loss / max(total, 1), 100 * correct / max(total, 1)

    def predict(self, data_loader):
        self.model.eval()
        all_logits = []

        with torch.no_grad():
            for batch in data_loader:
                input_ids = batch['input_ids'].to(self.device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(self.device, non_blocking=True)
                logits = self.model(input_ids, attention_mask)
                all_logits.append(logits.cpu())

        return torch.cat(all_logits, dim=0)

print("Ensemble worker class defined")

In [ ]:
# Create ensemble workers
def resolve_worker_devices(config):
    if config.num_gpus > 0:
        return [torch.device(f"cuda:{i % config.num_gpus}") for i in range(config.num_workers)]
    return [torch.device("cpu") for _ in range(config.num_workers)]


workers = []
devices = resolve_worker_devices(config)

print(f"Creating {config.num_workers} ensemble workers:\n")
for i, device in enumerate(devices):
    worker = EnsembleWorker(i, config, device)
    workers.append(worker)

print(f"\n{config.num_workers} workers created")

In [ ]:
# Training loop
print(f"Starting training for {config.num_epochs} epochs\n")

best_ensemble_acc = 0.0
num_training_steps = max(len(train_loader) * config.num_epochs, 1)
num_warmup_steps = int(num_training_steps * config.warmup_ratio)

schedulers = {
    worker.worker_id: get_cosine_schedule_with_warmup(
        worker.optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
    )
    for worker in workers
}

for epoch in range(config.num_epochs):
    print(f"\n{'='*80}")
    print(f"Epoch {epoch + 1}/{config.num_epochs}")
    print(f"{'='*80}")

    # Train each worker
    for worker in workers:
        scheduler = schedulers[worker.worker_id]
        train_loss, train_acc = worker.train_epoch(train_loader, scheduler)
        val_loss, val_acc = worker.evaluate(val_loader)

        worker.train_losses.append(train_loss)
        worker.val_losses.append(val_loss)
        worker.val_accuracies.append(val_acc)

        print(f"\nWorker {worker.worker_id}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")

    # Ensemble evaluation
    print(f"\nEnsemble Evaluation:")
    all_worker_logits = []

    for worker in workers:
        logits = worker.predict(val_loader)
        all_worker_logits.append(logits)

    ensemble_logits = torch.stack(all_worker_logits).mean(dim=0)
    ensemble_preds = torch.argmax(ensemble_logits, dim=1).numpy()
    true_labels = val_df['label'].values

    ensemble_acc = accuracy_score(true_labels, ensemble_preds) * 100
    ensemble_f1 = f1_score(true_labels, ensemble_preds, average='weighted') * 100

    print(f"  Ensemble Accuracy: {ensemble_acc:.2f}%")
    print(f"  Ensemble F1-Score: {ensemble_f1:.2f}%")

    if ensemble_acc > best_ensemble_acc:
        best_ensemble_acc = ensemble_acc
        print("  New best!")

print(f"\nTraining completed! Best validation accuracy: {best_ensemble_acc:.2f}%")

In [ ]:
# Final evaluation on test set
print("\nFinal Evaluation on Test Set\n")

test_worker_logits = []
for worker in workers:
    logits = worker.predict(test_loader)
    test_worker_logits.append(logits)

ensemble_test_logits = torch.stack(test_worker_logits).mean(dim=0)
ensemble_test_preds = torch.argmax(ensemble_test_logits, dim=1).numpy()
test_true_labels = test_df['label'].values

test_acc = accuracy_score(test_true_labels, ensemble_test_preds) * 100
test_f1 = f1_score(test_true_labels, ensemble_test_preds, average='weighted') * 100

print(f"{'='*80}")
print(f"FINAL TEST RESULTS")
print(f"{'='*80}")
print(f"Ensemble Test Accuracy: {test_acc:.2f}%")
print(f"Ensemble Test F1-Score: {test_f1:.2f}%")

print(f"\nClassification Report:")
print(classification_report(test_true_labels, ensemble_test_preds, target_names=config.distortion_names, digits=3))

# Confusion matrix
cm = confusion_matrix(test_true_labels, ensemble_test_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f"{i}" for i in range(10)],
            yticklabels=[f"{i}" for i in range(10)])
plt.title('Confusion Matrix - DeBERTa HQDE Ensemble')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(f'{config.output_dir}/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Confusion matrix saved")

# Per-class accuracy
print(f"\nPer-Class Accuracy:")
for i in range(config.num_classes):
    class_mask = test_true_labels == i
    if class_mask.sum() > 0:
        class_acc = (ensemble_test_preds[class_mask] == test_true_labels[class_mask]).mean() * 100
        print(f"  {i}: {config.distortion_names[i]:35s} - {class_acc:.2f}%")

# Individual worker performance
print(f"\nIndividual Worker Test Accuracy:")
for i, worker in enumerate(workers):
    worker_logits = test_worker_logits[i]
    worker_preds = torch.argmax(worker_logits, dim=1).numpy()
    worker_acc = accuracy_score(test_true_labels, worker_preds) * 100
    print(f"  Worker {i}: {worker_acc:.2f}%")

print(f"\n{'='*80}")
print(f"✅ COMPLETE!")
print(f"{'='*80}")
print(f"Best Validation Accuracy: {best_ensemble_acc:.2f}%")
print(f"Final Test Accuracy: {test_acc:.2f}%")
print(f"Final Test F1-Score: {test_f1:.2f}%")